In [1]:
import json
import pickle
from pathlib import Path

import numpy as np

import sys
sys.path.append("../workflow/scripts")
from helpers import fit_clorinn, ensure_parent
from cv_sr_fit_clorinn_common import load_split_input, load_zscore_and_noise

In [2]:
input_dir = "/gpfs/commons/groups/knowles_lab/data/PsychGen"
cv_input       = Path(input_dir) / "analysis/clorinn/cv_split_replication/cv_input/split_assignments.npz"
zscore_data    = Path(input_dir) / "input/pgc_v1.0/zscore_v1_0.csv"
noise_cov_data = Path(input_dir) / "input/pgc_v1.0/sampling_covariance_v1_0.csv"
method = "pgd-afw"
model  = "nnm-corr"

assignments, eligible_cols, fold_sizes, p_total, n_folds, n_repeats = (
    load_split_input(cv_input)
)
Z, noise_cov = load_zscore_and_noise(
    zscore_data,
    noise_cov_path = noise_cov_data,
    model = model
)

In [16]:
fold_id = 0
repeat_id = 0
nucnorm_full = 32768
col_idx    = eligible_cols[assignments[repeat_id] == fold_id]
n_cols_fit = col_idx.size
r_fit      = nucnorm_full * np.sqrt(n_cols_fit / eligible_cols.size)

print(
    f"  fold {fold_id}: n_cols={n_cols_fit}, r_fit={r_fit:.4f}, "
    f"scale={np.sqrt(n_cols_fit / eligible_cols.size):.6f}"
)

Z_fold = Z[:, col_idx]

  fold 0: n_cols=23252, r_fit=23170.4750, scale=0.707107


In [17]:
Z_fold.shape

(107, 23252)

In [18]:
noise_cov.shape

(107, 107)

In [20]:
from clorinn.optimize import ProjectedGradientDescent
pgd = ProjectedGradientDescent(model = model, max_iter = 100, rel_tol=1e-4, verbose = 2)
pgd.fit(Z_fold, radius = r_fit, noise_cov = noise_cov)

2026-05-11 10:01:48,574 | pgd                      | DEBUG   | Model = nnm-corr
2026-05-11 10:01:48,576 | pgd                      | DEBUG   | Shape of input data = (107, 23252)
2026-05-11 10:01:49,123 | objectives               | DEBUG   | NNMCorrObjective: shape=(107, 23252), radius=23170.47500592079, masked_entries=123807, weighted=no
2026-05-11 10:01:49,175 | noise_cov_op             | DEBUG   | CovarianceOperator: n=107, lambda_min(A)=0.01526, lambda_max(A)=19.2, cond(A)=1258, pgd_step_size=0.01526
2026-05-11 10:01:55,661 | objectives               | DEBUG   | NNMCorrObjective: pgd_step_size=0.01526, n_patterns=3400
2026-05-11 10:01:55,666 | pgd                      | DEBUG   | Masked entries = 123807
2026-05-11 10:01:56,972 | pgd                      | INFO    | PGD iter    1  f = 2415439.5377  ||X||_* = 495.4  (interior)
2026-05-11 10:01:57,830 | pgd                      | INFO    | PGD iter    2  f = 2285922.8595  ||X||_* = 911.1  (interior)
2026-05-11 10:01:58,683 | pgd       

In [23]:
from clorinn.optimize import AwayStepFrankWolfe
afw = AwayStepFrankWolfe(model = model, max_iter = 1000, svd_method = "left-gram", verbose = 1, tol = 100, stop_criteria=("duality_gap",))
afw.fit(Z_fold, radius = r_fit, noise_cov = noise_cov, X0 = pgd.result.X)

2026-05-11 10:06:15,634 | frankwolfe               | INFO    | Iteration 1. Step size 0.013. Duality Gap 2.54168e+06
2026-05-11 10:07:37,122 | frankwolfe               | INFO    | Iteration 100. Step size 0.000. Duality Gap 3.05712e+06
2026-05-11 10:09:01,502 | frankwolfe               | INFO    | Iteration 200. Step size 0.000. Duality Gap 906749
2026-05-11 10:10:26,426 | frankwolfe               | INFO    | Iteration 300. Step size 0.000. Duality Gap 778814
2026-05-11 10:11:52,155 | frankwolfe               | INFO    | Iteration 400. Step size 0.001. Duality Gap 573209
2026-05-11 10:13:17,704 | frankwolfe               | INFO    | Iteration 500. Step size 0.000. Duality Gap 576706
2026-05-11 10:14:44,437 | frankwolfe               | INFO    | Iteration 600. Step size 0.000. Duality Gap 723384
2026-05-11 10:16:12,120 | frankwolfe               | INFO    | Iteration 700. Step size 0.000. Duality Gap 538280
2026-05-11 10:17:39,986 | frankwolfe               | INFO    | Iteration 800. St

In [5]:
np.linalg.norm(pgd.result.X, ord='nuc')

8192.000000000038

In [6]:
from clorinn import MatrixFactorization
mf = MatrixFactorization(k = 20)
mf.fit(pgd.result.X)
s2 = mf.s**2
np.cumsum(s2)/np.sum(s2)

array([0.22023086, 0.42794757, 0.56123994, 0.63701227, 0.7041738 ,
       0.76152097, 0.81022333, 0.85230446, 0.88382079, 0.90649784,
       0.92599778, 0.94013692, 0.95178969, 0.96299306, 0.97043273,
       0.97690343, 0.98187319, 0.98618147, 0.98946784, 0.99221243,
       0.9947922 , 0.9965341 , 0.99769811, 0.99861584, 0.9989864 ,
       0.99930556, 0.99951494, 0.99968973, 0.9997914 , 0.99987594,
       0.99995794, 0.99998623, 0.99999808, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.     

In [8]:
np.sum(np.linalg.svd(afw.result.X, compute_uv=False))

8191.999999333758

In [9]:
from clorinn import MatrixFactorization
mf = MatrixFactorization(k = 20)
mf.fit(pgd.result.X)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print ("X0 after PGD")
print (energy[:20])

mf = MatrixFactorization(k = 20)
mf.fit(afw.result.X)
s2 = mf.s**2
energy = np.cumsum(s2) / np.sum(s2)
print ("AFW after PGD")
print (energy[:20])

X0 after PGD
[0.22023086 0.42794757 0.56123994 0.63701227 0.7041738  0.76152097
 0.81022333 0.85230446 0.88382079 0.90649784 0.92599778 0.94013692
 0.95178969 0.96299306 0.97043273 0.97690343 0.98187319 0.98618147
 0.98946784 0.99221243]
FW after PGD
[0.2202654  0.42797568 0.56126874 0.63704652 0.70420852 0.76155294
 0.81025935 0.85234177 0.88385486 0.90652975 0.92602469 0.94015967
 0.95180822 0.96300848 0.97044573 0.97691322 0.98188152 0.98618771
 0.98947275 0.99221621]


In [10]:
print(f"PGD objective:         {pgd.result.loss:.2f}")
print(f"AFW after 1000 steps:  {afw.result.loss:.2f}")
print(f"Relative improvement:  {(pgd.result.loss - afw.result.loss) / pgd.result.loss:g}")

PGD objective:         2195713.31
FW after 1000 steps:   2195713.28
Relative improvement:  1.4628e-08
